# E2E Usecase Metrics Report

Notebook nay doc cac file JSON trong `testing-e2e/artifacts/usecases` va thong ke latency, score, failed steps, failure reasons, tool usage va tool errors.

Chay tung cell tu tren xuong. Neu artifact moi duoc sinh ra, chay lai notebook de cap nhat bang thong ke.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from statistics import mean

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda text: text

def find_artifact_dir() -> Path:
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates = [
            base / "artifacts" / "usecases",
            base / "testing-e2e" / "artifacts" / "usecases",
        ]
        for candidate in candidates:
            if candidate.exists() and candidate.is_dir():
                return candidate
    raise FileNotFoundError("Cannot find testing-e2e/artifacts/usecases from current working directory")

ARTIFACT_DIR = find_artifact_dir()
SUMMARY_PATH = ARTIFACT_DIR / "agent-evaluation-report.json"

ARTIFACT_DIR

## Load and normalize artifacts

Latency is computed from `durationMs` when available, otherwise from `finishedAt - startedAt`. Current artifacts have usecase-level timestamps, so per-step latency below is an average estimate: `usecase latency / number of steps`.

In [ ]:
def load_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def parse_dt(value):
    if not value:
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(value, tz=timezone.utc)
    text = str(value).strip()
    if not text:
        return None
    if text.endswith("Z"):
        text = text[:-1] + "+00:00"
    try:
        return datetime.fromisoformat(text)
    except ValueError:
        return None

def duration_seconds(report: dict) -> float | None:
    duration_ms = report.get("durationMs")
    if isinstance(duration_ms, (int, float)):
        return round(duration_ms / 1000, 3)
    started = parse_dt(report.get("startedAt"))
    finished = parse_dt(report.get("finishedAt"))
    if started and finished:
        return round((finished - started).total_seconds(), 3)
    return None

def as_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False)

def shorten(value, limit=260) -> str:
    text = as_text(value).replace("\r", " ").replace("\n", " ").strip()
    if len(text) <= limit:
        return text
    return text[: limit - 3] + "..."

def to_frame(rows):
    if pd is None:
        return rows
    return pd.DataFrame(rows)

def show_table(rows, columns=None, sort_by=None, ascending=True):
    if pd is None:
        display(rows)
        return rows
    df = pd.DataFrame(rows)
    if sort_by and sort_by in df.columns:
        df = df.sort_values(sort_by, ascending=ascending, na_position="last")
    if columns:
        df = df[[col for col in columns if col in df.columns]]
    display(df)
    return df

def iter_tool_traces(step: dict):
    agent = step.get("agent") if isinstance(step.get("agent"), dict) else {}
    for trace in agent.get("toolTrace") or []:
        if isinstance(trace, dict):
            yield trace

run_paths = sorted(path for path in ARTIFACT_DIR.glob("*.json") if path.name != SUMMARY_PATH.name)
runs = []
for path in run_paths:
    report = load_json(path)
    if isinstance(report, dict) and "conversation" in report:
        runs.append({"path": path, "report": report})

summary = load_json(SUMMARY_PATH) if SUMMARY_PATH.exists() else {}
eval_by_usecase = {
    str(item.get("usecase")): item
    for item in summary.get("usecases", [])
    if isinstance(item, dict) and item.get("usecase")
}

display(Markdown(f"Loaded **{len(runs)}** run artifact(s) from `{ARTIFACT_DIR}`. Evaluation summary exists: **{SUMMARY_PATH.exists()}**"))

In [ ]:
usecase_rows = []
step_rows = []
tool_rows = []

for item in runs:
    path = item["path"]
    report = item["report"]
    usecase = str(report.get("usecase") or path.stem)
    eval_report = eval_by_usecase.get(usecase, {})
    eval_steps = {
        int(step.get("step")): step
        for step in eval_report.get("steps", [])
        if isinstance(step, dict) and isinstance(step.get("step"), int)
    }
    conversation = report.get("conversation") or []
    latency_s = duration_seconds(report)
    step_count = len(conversation)
    failed_steps = eval_report.get("failedSteps")
    if not isinstance(failed_steps, list):
        failed_steps = [step.get("step") for step in conversation if step.get("passed") is False]

    tool_counter = Counter()
    tool_error_count = 0
    approval_count = 0
    for step in conversation:
        agent = step.get("agent") if isinstance(step.get("agent"), dict) else {}
        tools = agent.get("tools") or []
        tool_counter.update(str(tool) for tool in tools)
        for trace in iter_tool_traces(step):
            tool_name = str(trace.get("toolName") or "")
            phase = str(trace.get("phase") or "")
            success = trace.get("success")
            if phase == "approval_requested":
                approval_count += 1
            is_error = success is False or "error" in phase.lower() or bool(trace.get("error"))
            if is_error:
                tool_error_count += 1
            tool_rows.append({
                "usecase": usecase,
                "step": step.get("step"),
                "tool": tool_name,
                "phase": phase,
                "success": success,
                "is_error": is_error,
                "toolCallId": trace.get("toolCallId"),
                "error": shorten(trace.get("error") or trace.get("message") or trace.get("data"), 400),
            })

    score = eval_report.get("score")
    if score is None and step_count:
        score = sum(1 for step in conversation if step.get("passed") is True) / step_count

    usecase_rows.append({
        "usecase": usecase,
        "status": eval_report.get("status") or ("passed" if report.get("passed") is True else "failed"),
        "sourcePassed": eval_report.get("sourcePassed", report.get("passed")),
        "score": score,
        "step_count": step_count,
        "failed_step_count": len([step for step in failed_steps if step is not None]),
        "failed_steps": failed_steps,
        "latency_s": latency_s,
        "latency_min": round(latency_s / 60, 3) if latency_s is not None else None,
        "avg_step_latency_s": round(latency_s / step_count, 3) if latency_s is not None and step_count else None,
        "tool_call_count": sum(tool_counter.values()),
        "unique_tools": ", ".join(sorted(tool_counter)),
        "approval_count": approval_count,
        "tool_error_count": tool_error_count,
        "startedAt": report.get("startedAt"),
        "finishedAt": report.get("finishedAt"),
        "runId": report.get("runId"),
        "file": path.name,
    })

    for step in conversation:
        step_id = step.get("step")
        agent = step.get("agent") if isinstance(step.get("agent"), dict) else {}
        user = step.get("user") if isinstance(step.get("user"), dict) else {}
        eval_step = eval_steps.get(step_id, {}) if isinstance(step_id, int) else {}
        llm_eval = eval_step.get("llmEvaluation") if isinstance(eval_step.get("llmEvaluation"), dict) else {}
        trace_errors = [trace for trace in iter_tool_traces(step) if trace.get("success") is False or "error" in str(trace.get("phase", "")).lower() or trace.get("error")]
        step_rows.append({
            "usecase": usecase,
            "step": step_id,
            "run_step_passed": step.get("passed"),
            "eval_passed": llm_eval.get("passed"),
            "agent_status": agent.get("status"),
            "approvalTool": agent.get("approvalTool"),
            "tools": ", ".join(str(tool) for tool in (agent.get("tools") or [])),
            "tool_error_count": len(trace_errors),
            "failureReason": eval_step.get("failureReason") or llm_eval.get("reason") if llm_eval.get("passed") is False else eval_step.get("failureReason"),
            "userMessage": shorten(user.get("message"), 220),
            "agentMessage": shorten(agent.get("message"), 320),
            "expectedAgent": shorten(eval_step.get("expectedAgent"), 320),
            "judge_model": llm_eval.get("model"),
        })

usecases_df = to_frame(usecase_rows)
steps_df = to_frame(step_rows)
tools_df = to_frame(tool_rows)

len(usecase_rows), len(step_rows), len(tool_rows)

## Overview

In [ ]:
if pd is None:
    total = len(usecase_rows)
    passed = sum(1 for row in usecase_rows if row["status"] == "passed")
    failed = total - passed
    avg_score = mean([row["score"] for row in usecase_rows if row["score"] is not None]) if usecase_rows else None
    avg_latency = mean([row["latency_s"] for row in usecase_rows if row["latency_s"] is not None]) if usecase_rows else None
    overview = {"total": total, "passed": passed, "failed": failed, "pass_rate": passed / total if total else None, "avg_score": avg_score, "avg_latency_s": avg_latency}
    display(overview)
else:
    total = len(usecases_df)
    overview = pd.DataFrame([{
        "total_usecases": total,
        "passed": int((usecases_df["status"] == "passed").sum()) if total else 0,
        "failed": int((usecases_df["status"] != "passed").sum()) if total else 0,
        "pass_rate": round(float((usecases_df["status"] == "passed").mean()), 4) if total else None,
        "avg_score": round(float(usecases_df["score"].dropna().mean()), 4) if total and usecases_df["score"].notna().any() else None,
        "total_latency_s": round(float(usecases_df["latency_s"].dropna().sum()), 3) if total else None,
        "avg_latency_s": round(float(usecases_df["latency_s"].dropna().mean()), 3) if total and usecases_df["latency_s"].notna().any() else None,
        "p95_latency_s": round(float(usecases_df["latency_s"].dropna().quantile(0.95)), 3) if total and usecases_df["latency_s"].notna().any() else None,
        "failed_steps": int(usecases_df["failed_step_count"].sum()) if total else 0,
        "tool_errors": int(usecases_df["tool_error_count"].sum()) if total else 0,
    }])
    display(overview)

## Usecase metrics

In [ ]:
usecase_columns = [
    "usecase", "status", "score", "latency_s", "avg_step_latency_s", "step_count",
    "failed_step_count", "failed_steps", "tool_call_count", "approval_count", "tool_error_count", "unique_tools", "file"
]
show_table(usecase_rows, columns=usecase_columns, sort_by="score", ascending=True)

## Failed steps and wrong answers

Bang nay tap trung vao cac loi sai: failed evaluation, failed raw step, hoac tool trace co error.

In [ ]:
if pd is None:
    failed_step_rows = [
        row for row in step_rows
        if row.get("eval_passed") is False or row.get("run_step_passed") is False or row.get("tool_error_count", 0) > 0 or row.get("failureReason")
    ]
else:
    failed_step_rows = steps_df[
        (steps_df["eval_passed"] == False)
        | (steps_df["run_step_passed"] == False)
        | (steps_df["tool_error_count"] > 0)
        | (steps_df["failureReason"].fillna("") != "")
    ].to_dict("records")

failed_columns = [
    "usecase", "step", "eval_passed", "run_step_passed", "agent_status", "tools",
    "failureReason", "userMessage", "agentMessage", "expectedAgent"
]
show_table(failed_step_rows, columns=failed_columns)

## All steps

In [ ]:
step_columns = [
    "usecase", "step", "eval_passed", "run_step_passed", "agent_status", "approvalTool", "tools", "tool_error_count", "failureReason", "judge_model"
]
show_table(step_rows, columns=step_columns)

## Tool usage and tool errors

In [ ]:
if pd is None:
    tool_usage = Counter(row["tool"] for row in tool_rows if row.get("tool"))
    display(tool_usage)
else:
    if tools_df.empty:
        display(pd.DataFrame(columns=["tool", "count", "error_count", "approval_requested_count"]))
    else:
        tool_usage = (
            tools_df.groupby("tool", dropna=False)
            .agg(
                count=("tool", "size"),
                error_count=("is_error", "sum"),
                approval_requested_count=("phase", lambda s: int((s == "approval_requested").sum())),
            )
            .reset_index()
            .sort_values(["error_count", "count"], ascending=[False, False])
        )
        display(tool_usage)

if pd is not None and not tools_df.empty:
    display(tools_df[tools_df["is_error"] == True][["usecase", "step", "tool", "phase", "success", "error"]])

## Charts

In [ ]:
if pd is None or plt is None:
    display(Markdown("Install `pandas` and `matplotlib` to render charts."))
elif usecases_df.empty:
    display(Markdown("No usecase rows to plot."))
else:
    plot_df = usecases_df.sort_values("score", ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, max(4, len(plot_df) * 0.45)))

    axes[0].barh(plot_df["usecase"], plot_df["score"].fillna(0), color="#2f6fdd")
    axes[0].set_title("Score by usecase")
    axes[0].set_xlim(0, 1)
    axes[0].set_xlabel("score")

    axes[1].barh(plot_df["usecase"], plot_df["latency_s"].fillna(0), color="#d46b08")
    axes[1].set_title("Latency by usecase")
    axes[1].set_xlabel("seconds")

    plt.tight_layout()
    plt.show()

## Export tables

Optional: chay cell duoi de xuat cac bang CSV vao `testing-e2e/artifacts/usecases/metrics_exports`.

In [ ]:
EXPORT = False

if EXPORT:
    if pd is None:
        raise RuntimeError("CSV export requires pandas")
    export_dir = ARTIFACT_DIR / "metrics_exports"
    export_dir.mkdir(parents=True, exist_ok=True)
    usecases_df.to_csv(export_dir / "usecase_metrics.csv", index=False, encoding="utf-8-sig")
    steps_df.to_csv(export_dir / "step_metrics.csv", index=False, encoding="utf-8-sig")
    tools_df.to_csv(export_dir / "tool_traces.csv", index=False, encoding="utf-8-sig")
    display(Markdown(f"Exported CSV files to `{export_dir}`"))
else:
    display(Markdown("Set `EXPORT = True` and run this cell to write CSV files."))